# Analyzing Experiment Results
This code is intended to use results in the `saved_selection_data.csv` and `saved_yolked_data.csv` to generate plots and otherwise interpret the results of the experiments run.

In [3]:
from dataclasses import dataclass
from pathlib import Path
import ast
import pandas as pd
import matplotlib as plt

In [4]:
@dataclass(frozen=True,slots=True)
class experiment_metadata:
    UUID: str
    "A UUID to identify the experiment"
    category_boundary: str
    "The boundary between the two categories in the experiment (save vs. not)"
    learning_stimuli: list[list[float]]
    "All stimuli selected from learning. This is a float as a string."
    testing_stimuli: list[list[float]]
    "All stimuli used for testing. This is a float as a string. First axis should be the same size as learning_stimuli"
    testing_individual_results: list[list[bool]]
    "The categorizations made by the participant for each of the corresponting stimuli. <br>True='safe'=`> category boundary`  <br>False='not safe' = `< category boundary`"
    testing_individual_correctness: list[list[bool]]
    "Whether the individual correctly classified the testing stimuli. If True they categorized correctly."
    testing_round_correctness: list[float]
    "Combines correctness by round, giving the percentage correct"
    testing_round_certainties: list[int]
    "Reports the participant certainty ratings (0-4) by round"
    testing_overall_correctness: float
    "Combines correctness for the entirety of all tests, giving overall percentage correct"

def load_experiments(
    selection_path: Path,
    yolked_path: Path,
) -> tuple[list[experiment_metadata], list[experiment_metadata]]:
    def _parse_row(row) -> experiment_metadata:
        return experiment_metadata(
            UUID=row["UUID"],
            category_boundary=str(row["category_boundary"]),
            learning_stimuli=ast.literal_eval(row["learning_stimuli"]),
            testing_stimuli=ast.literal_eval(row["testing_stimuli"]),
            testing_individual_results=ast.literal_eval(row["testing_individual_results"]),
            testing_individual_correctness=ast.literal_eval(row["testing_individual_correctness"]),
            testing_round_correctness=ast.literal_eval(row["testing_round_correctness"]),
            testing_round_certainties=ast.literal_eval(row["testing_round_certainties"]),
            testing_overall_correctness=float(row["testing_overall_correctness"]),
        )

    selection_df = pd.read_csv(selection_path)
    yolked_df = pd.read_csv(yolked_path)

    return (
        [_parse_row(row) for _, row in selection_df.iterrows()],
        [_parse_row(row) for _, row in yolked_df.iterrows()],
    )


In [ ]:
selection_data, yolked_data = load_experiments(
    selection_path=Path("./saved_selection_data.csv"),
    yolked_path=Path("./saved_yolked_data.csv")
    )

print(selection_data[0])
print(selection_data[0].testing_round_certainties)
print([c+1 for c in selection_data[0].testing_round_certainties]) # Use list comprehension to increment all entries in a list
print([s.testing_round_certainties for s in selection_data]) # Use list comprehension to extract all certainties into one 2D list.

experiment_metadata(UUID='b4e7fda9-63aa-4f7e-967d-55fdc32ccaae', category_boundary='0.78', learning_stimuli=[[0.7800000000000002, 0.5], [0.7800000000000002, 0.6800000000000002], [0.6800000000000002, 0.9000000000000004]], testing_stimuli=[[0.6050390448342762, 0.2298115466716064], [0.7476321283105172, 0.5284251651318078], [0.9115910624183958, 0.0791704823870378]], testing_individual_results=[[True, False], [True, False], [True, False]], testing_individual_correctness=[[False, True], [False, True], [True, True]], testing_round_correctness=[0.5, 0.5, 1.0], testing_round_certainties=[0, 4, 2], testing_overall_correctness=0.6666666666666666)
[0, 4, 2]
[1, 5, 3]
[[0, 4, 2]]
